# Estrutura da Página HTML para RPA

## Link (GitHub Pages)

https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n

## Código HTML

```html
<!doctype html>
<html lang="pt-br">
<head>
  <meta charset="utf-8" />
  <meta name="viewport" content="width=device-width, initial-scale=1" />
  <title>Clientes</title>
  <style>
    body { font-family: Arial, sans-serif; margin: 24px; }
    table { border-collapse: collapse; width: 100%; max-width: 900px; }
    th, td { border: 1px solid #ddd; padding: 10px; text-align: left; }
    th { background: #f5f5f5; }
  </style>
</head>
<body>
  <h1>Lista de Clientes</h1>

  <table id="clientes">
    <thead>
      <tr>
        <th>Nome</th>
        <th>Email</th>
        <th>Saldo</th>
        <th>Perfil</th>
      </tr>
    </thead>
    <tbody>
      <tr>
        <td class="nome">Ana Silva</td>
        <td class="email">ana@email.com</td>
        <td class="saldo">R$ 12.500,00</td>
        <td class="perfil">Conservador</td>
      </tr>
      <tr>
        <td class="nome">Bruno Lima</td>
        <td class="email">bruno@email.com</td>
        <td class="saldo">R$ 3.200,00</td>
        <td class="perfil">Moderado</td>
      </tr>
      <tr>
        <td class="nome">Carla Souza</td>
        <td class="email">carla@email.com</td>
        <td class="saldo">R$ 28.900,00</td>
        <td class="perfil">Arrojado</td>
      </tr>
    </tbody>
  </table>
</body>
</html>
```

# Dependências (Python)

In [ ]:
!pip install requests beautifulsoup4

# Código-fonte RPA (Pyhton)

In [32]:
import re
import requests
from bs4 import BeautifulSoup

URL_PAGINA = "https://digitalinnovationone.github.io/dio-lab-assistente-investimentos-rpa-n8n/"
N8N_WEBHOOK = "https://andersonssantos.app.n8n.cloud/webhook-test/clientes"

# 1) Baixar a página HTML
response = requests.get(URL_PAGINA, timeout=30)
response.raise_for_status()

# 2) Extrair os clientes da tabela via BeautifulSoup
soup = BeautifulSoup(response.text, "html.parser")
linhas = soup.select("#clientes tbody tr")

clientes = []

for linha in linhas:
    colunas = linha.find_all("td")
    if len(colunas) >= 4:
        raw_saldo = colunas[2].get_text(strip=True)

        # Trata o saldo para converter em float (ex: "R$ 10.000,00" -> 10000.0)
        saldo_limpo = re.sub(r"[^\d,]", "", raw_saldo).replace(",", ".")
        try:
            saldo_val = float(saldo_limpo)
        except ValueError:
            saldo_val = raw_saldo

        cliente = {
            "nome": colunas[0].get_text(strip=True),
            "email": colunas[1].get_text(strip=True),
            "saldo": saldo_val,
            "perfil": colunas[3].get_text(strip=True),
        }
        clientes.append(cliente)

print(f"✅ Clientes encontrados: {len(clientes)}")
for c in clientes:
    print(c)

# 3) Enviar a lista de clientes para o Webhook do n8n via POST
payload = {"clientes": clientes}

resp = requests.post(
    N8N_WEBHOOK,
    json=payload,
    headers={"Content-Type": "application/json"},
    timeout=60,
)

print(f"Status do envio ao n8n: {resp.status_code}")
if resp.status_code == 200:
    print("Sucesso! Os dados foram recebidos pelo n8n.")
else:
    print(f"Atenção: O servidor retornou {resp.status_code}. Detalhes: {resp.text}")

✅ Clientes encontrados: 10
{'nome': 'Ana Silva', 'email': 'ana@email.com', 'saldo': 12500.0, 'perfil': 'Conservador'}
{'nome': 'Bruno Lima', 'email': 'bruno@email.com', 'saldo': 3200.0, 'perfil': 'Moderado'}
{'nome': 'Carla Souza', 'email': 'carla@email.com', 'saldo': 28900.0, 'perfil': 'Arrojado'}
{'nome': 'Daniel Oliveira', 'email': 'daniel@email.com', 'saldo': 850.0, 'perfil': 'Conservador'}
{'nome': 'Elena Rodrigues', 'email': 'elena@email.com', 'saldo': 45000.0, 'perfil': 'Arrojado'}
{'nome': 'Felipe Santos', 'email': 'felipe@email.com', 'saldo': 7800.0, 'perfil': 'Moderado'}
{'nome': 'Gabriela Costa', 'email': 'gabriela@email.com', 'saldo': 1500.0, 'perfil': 'Conservador'}
{'nome': 'Henrique Alves', 'email': 'henrique@email.com', 'saldo': 22300.0, 'perfil': 'Moderado'}
{'nome': 'Isabela Martins', 'email': 'isabela@email.com', 'saldo': 500.0, 'perfil': 'Conservador'}
{'nome': 'João Pereira', 'email': 'joao@email.com', 'saldo': 15000.0, 'perfil': 'Arrojado'}
Status do envio ao n8n: